# Tuần 4 — Huấn luyện `MedSAM-LoRA (r = 8)` & Phân vùng Khối u Não

**Thành viên phụ trách:** Huỳnh Nguyễn Nhân Quyết

### Hai nhiệm vụ thực nghiệm Tuần 4:
1. **Nhiệm vụ 1:** Xây dựng **`MedSAM-LoRA`** — đóng băng `Image Encoder (ViT-B)` và `Prompt Encoder`, chèn **LoRA (`r = 8, α = 16`)** vào `Query/Value` của 12 khối Attention, huấn luyện đầy đủ **`Mask Decoder`** (`4,63%` tham số khả huấn).
2. **Nhiệm vụ 2:** Cài đặt hàm mất mát $L_{\text{Seg}} = 0.5\,L_{\text{Dice}} + 0.5\,L_{\text{Focal/BCE}}$, áp dụng **Box Jittering `±10px`** (hệ `1024×1024`), huấn luyện và đánh giá cải thiện **`Dice`**, **`IoU`**, **`HD95`** so với **MedSAM Zero-shot**.


---
## Cell 1: Mount Google Drive & Cấu hình Đường dẫn / Siêu tham số


In [ ]:
from pathlib import Path
import os
import sys
import torch

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

# ================= CẤU HÌNH TRUNG TÂM =================
REPO_URL = "https://github.com/MikanQuyt/BrainTumorDetect.git"
REPO_BRANCH = "dev"

# Chọn 1 hoặc nhiều bộ dữ liệu có nhãn Mask: "brisc2025" | "indk214" | "brats2020"
TARGET_DATASETS = ["brisc2025"]

# Kiến trúc MedSAM-LoRA & hàm mất mát Tuần 4
LORA_R = 8
LORA_ALPHA = 16.0
LORA_DROPOUT = 0.05
LOSS_MODE = "focal"          # "focal" (0.5 Dice + 0.5 Focal) hoặc "bce" (0.5 Dice + 0.5 BCE)
DICE_WEIGHT = 0.5
PIXEL_WEIGHT = 0.5
BOX_JITTER = 10.0            # Biên độ Box Jittering ±10px trên hệ 1024x1024 (0.0 để tắt)

# Lịch huấn luyện
EPOCHS = 15
BATCH_SIZE = 2               # T4 16GB: dùng 2 + GRAD_ACCUM=8; L4/A100: có thể tăng lên 4 + GRAD_ACCUM=4
GRAD_ACCUM = 8
LR_LORA = 1e-4
LR_DECODER = 1e-4
WEIGHT_DECAY = 0.01
PATIENCE = 5
NUM_WORKERS = 2 if IN_COLAB else 0
SEED = 42

# Tùy chọn chạy thêm
RUN_SMOKE_TEST = False       # Bật True để chạy thử 16 mẫu / 1 epoch kiểm tra nhanh VRAM
RUN_LOSS_COMPARISON = False  # Bật True để chạy thêm cấu hình còn lại (Focal vs BCE) để đối sánh
EVAL_MAX_SAMPLES = 0         # 0 = đánh giá toàn bộ tập test; đặt ví dụ 200 nếu test BraTS2020 quá lớn

# Thư mục dự án cục bộ & thư mục lưu trữ bền vững trên Google Drive
PROJECT_ROOT = Path("/content/BrainTumorDetect") if IN_COLAB else Path.cwd().resolve()
if not IN_COLAB and PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DRIVE_WORKDIR = (
    Path("/content/drive/MyDrive/BrainTumorDetect_Week4")
    if IN_COLAB
    else PROJECT_ROOT / "runs" / "week4_colab_local"
)
DRIVE_CKPT_DIR = DRIVE_WORKDIR / "pretrained_checkpoints"
DRIVE_DATA_CACHE = DRIVE_WORKDIR / "data_cache"
DS_TAG = "_".join(TARGET_DATASETS)
RUN_NAME = f"medsam_lora_{DS_TAG}_r{LORA_R}_{LOSS_MODE}_j{int(BOX_JITTER)}"
RUN_DIR = DRIVE_WORKDIR / RUN_NAME

for d in (DRIVE_WORKDIR, DRIVE_CKPT_DIR, DRIVE_DATA_CACHE, RUN_DIR):
    d.mkdir(parents=True, exist_ok=True)

os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"GPU: {gpu_name} ({vram_gb:.1f} GB VRAM) | CUDA {torch.version.cuda}")
    if vram_gb >= 22 and BATCH_SIZE == 2:
        print("Gợi ý: GPU >= 22 GB VRAM, bạn có thể tăng BATCH_SIZE=4, GRAD_ACCUM=4 để train nhanh hơn.")
else:
    print("Cảnh báo: Chưa bật GPU! Hãy chọn Runtime -> Change runtime type -> GPU trước khi huấn luyện.")

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"RUN_DIR      = {RUN_DIR}")

---
## Cell 2: Clone Mã nguồn, Cài đặt Thư viện & Kiểm định Unit Tests


In [ ]:
import subprocess

if IN_COLAB:
    if not (PROJECT_ROOT / ".git").exists():
        subprocess.check_call(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(PROJECT_ROOT)])
    else:
        subprocess.check_call(["git", "fetch", "origin", REPO_BRANCH], cwd=PROJECT_ROOT)
        subprocess.check_call(["git", "checkout", REPO_BRANCH], cwd=PROJECT_ROOT)
        subprocess.check_call(["git", "pull", "--ff-only", "origin", REPO_BRANCH], cwd=PROJECT_ROOT)

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

if IN_COLAB:
    pkgs = [
        "git+https://github.com/facebookresearch/segment-anything.git",
        "albumentations>=2.0",
        "opencv-python-headless",
        "scipy",
        "scikit-learn",
        "kagglehub",
        "nibabel",
        "h5py",
        "matplotlib",
        "pandas",
        "tqdm",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

# Kiểm tra nhanh unit test MedSAM-LoRA + SegmentationLoss + Box Jittering
subprocess.check_call([sys.executable, "-m", "unittest", "tests.test_medsam_lora", "-v"])
print("Môi trường và mã nguồn MedSAM-LoRA sẵn sàng!")

---
## Cell 3: Tải Trọng số Gốc `medsam_vit_b.pth` & Xác thực MD5


In [ ]:
import shutil
from utils.checkpoints import CHECKPOINTS, checkpoint_path, download_checkpoint, md5sum, verify_checkpoint

local_ckpt = checkpoint_path("medsam")
drive_ckpt = DRIVE_CKPT_DIR / CHECKPOINTS["medsam"]["file"]
local_ckpt.parent.mkdir(parents=True, exist_ok=True)

if verify_checkpoint("medsam"):
    print(f"[OK] Đã có sẵn {local_ckpt} đúng MD5.")
elif drive_ckpt.is_file() and md5sum(drive_ckpt) == CHECKPOINTS["medsam"]["md5"]:
    print(f"Đang chép medsam_vit_b.pth từ cache Drive ({drive_ckpt}) về máy Colab...")
    shutil.copy2(drive_ckpt, local_ckpt)
else:
    print("Đang tải medsam_vit_b.pth từ Zenodo (có hỗ trợ resume & kiểm tra MD5)...")
    download_checkpoint("medsam", ckpt_dir=local_ckpt.parent)
    if IN_COLAB:
        shutil.copy2(local_ckpt, drive_ckpt)
        print(f"Đã lưu cache lên Drive: {drive_ckpt}")

assert verify_checkpoint("medsam"), "MD5 của checkpoints/medsam_vit_b.pth không khớp!"
size_mb = local_ckpt.stat().st_size / (1024 ** 2)
print(f"Sẵn sàng trọng số gốc: {local_ckpt} ({size_mb:.1f} MB | MD5={md5sum(local_ckpt)})")

---
## Cell 4: Tải & Chuẩn hóa Dữ liệu có nhãn Mask (`brisc2025` / `indk214` / `brats2020`)


In [ ]:
import json
import pandas as pd

DATASET_SPECS = {
    "brisc2025": {
        "ann": "dataset/ds3_annotations.json",
        "processed_rel": "dataset/ds3_processed",
        "cache_zip": DRIVE_DATA_CACHE / "ds3_processed.zip",
        "kaggle_slug": "briscdataset/brisc2025",
    },
    "indk214": {
        "ann": "dataset/ds2_annotations.json",
        "processed_rel": "dataset/ds2_cropped",
        "cache_zip": DRIVE_DATA_CACHE / "ds2_cropped.zip",
        "kaggle_slug": "indk214/brain-tumor-dataset-segmentation-and-classification",
    },
    "brats2020": {
        "ann": "dataset/brats2020_annotations.json",
        "processed_rel": "dataset/brats2020_png",
        "cache_zip": DRIVE_DATA_CACHE / "brats2020_png.zip",
        "kaggle_slug": "awsaf49/brats20-dataset-training-validation",
    },
}


def _mask_records_for(ann_file: Path) -> list[dict]:
    recs = json.loads(ann_file.read_text(encoding="utf-8-sig"))
    return [r for r in recs if bool(r.get("has_mask")) and bool(r.get("mask_path")) and int(r.get("has_tumor", 0)) == 1]


def _link_or_copy(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        return
    try:
        os.symlink(src, dst, target_is_directory=True)
    except OSError:
        shutil.copytree(src, dst, dirs_exist_ok=True)


def prepare_mask_dataset(name: str):
    spec = DATASET_SPECS[name]
    ann_path = PROJECT_ROOT / spec["ann"]
    recs = _mask_records_for(ann_path)
    assert recs, f"Không tìm thấy mẫu có mask trong {ann_path}"
    sample_img = PROJECT_ROOT / recs[0]["img_path"]
    sample_msk = PROJECT_ROOT / recs[0]["mask_path"]

    if not (sample_img.exists() and sample_msk.exists()):
        cache_zip = spec["cache_zip"]
        if IN_COLAB and cache_zip.is_file():
            print(f"[{name}] Đang giải nén cache từ Drive: {cache_zip} ...")
            subprocess.check_call(["unzip", "-q", "-o", str(cache_zip), "-d", str(PROJECT_ROOT)])

    if not (sample_img.exists() and sample_msk.exists()):
        import kagglehub
        print(f"[{name}] Đang tải dữ liệu gốc từ Kaggle ({spec['kaggle_slug']})...")
        dl_path = Path(kagglehub.dataset_download(spec["kaggle_slug"]))
        ann_backup = ann_path.read_text(encoding="utf-8-sig")

        if name == "brisc2025":
            src_raw = dl_path / "brisc2025" if (dl_path / "brisc2025" / "classification_task").exists() else dl_path
            _link_or_copy(src_raw, PROJECT_ROOT / "dataset" / "brisc2025" / "brisc2025")
            subprocess.check_call([sys.executable, "-m", "data.build_ds3", "--root", str(PROJECT_ROOT)])
        elif name == "indk214":
            ind_cands = [dl_path / "DATASET"] + sorted(p for p in dl_path.rglob("DATASET") if p.is_dir())
            ind_base = next((p for p in ind_cands if (p / "Segmentation").is_dir()), dl_path)
            _link_or_copy(ind_base, PROJECT_ROOT / "dataset" / "indk214" / "DATASET")
            # Đảm bảo navoneel có thể trống nếu chỉ chạy indk214 — tải luôn navoneel nhỏ (~15MB) để build_ds1_ds2 chạy trọn
            nav_dl = Path(kagglehub.dataset_download("navoneel/brain-mri-images-for-brain-tumor-detection"))
            nav_cands = [nav_dl] + sorted(p.parent for p in nav_dl.rglob("yes") if p.is_dir())
            nav_base = next((p for p in nav_cands if (p / "yes").is_dir() and (p / "no").is_dir()), nav_dl)
            _link_or_copy(nav_base, PROJECT_ROOT / "dataset" / "navoneel")
            subprocess.check_call([sys.executable, "-m", "data.build_ds1_ds2"])
        elif name == "brats2020":
            brats_cands = [dl_path] + sorted(p.parent for p in dl_path.rglob("BraTS20_Training_001") if p.is_dir())
            brats_dir = brats_cands[0]
            subprocess.check_call([
                sys.executable, "data/build_dataset.py",
                "--data_dir", str(brats_dir),
                "--out_dir", str(PROJECT_ROOT / "dataset" / "brats2020_png"),
            ])

        # Khôi phục đúng file annotation chuẩn trên nhánh dev để giữ nguyên split
        ann_path.write_text(ann_backup, encoding="utf-8")

        if IN_COLAB and (PROJECT_ROOT / spec["processed_rel"]).is_dir():
            print(f"[{name}] Đang lưu cache lên Drive: {spec['cache_zip']} ...")
            subprocess.call(["zip", "-q", "-r", str(spec["cache_zip"]), spec["processed_rel"]])

    missing_img = [r["img_path"] for r in recs if not (PROJECT_ROOT / r["img_path"]).is_file()]
    missing_msk = [r["mask_path"] for r in recs if not (PROJECT_ROOT / r["mask_path"]).is_file()]
    assert not missing_img and not missing_msk, (
        f"[{name}] Thiếu {len(missing_img)} ảnh / {len(missing_msk)} mask! Ví dụ: {missing_img[:2]} {missing_msk[:2]}"
    )
    return recs


rows_summary = []
CLASS_NAMES = {0: "no_tumor", 1: "glioma", 2: "meningioma", 3: "pituitary"}
for ds_name in TARGET_DATASETS:
    recs = prepare_mask_dataset(ds_name)
    for r in recs:
        rows_summary.append({
            "dataset": ds_name,
            "split": r["split"],
            "class": CLASS_NAMES.get(int(r.get("sub_class_label", 1)), "tumor"),
        })

df_ds = pd.DataFrame(rows_summary)
print(f"Tổng số mẫu phân vùng hợp lệ (has_mask=True, has_tumor=1): {len(df_ds):,}")
display(pd.crosstab([df_ds["dataset"], df_ds["split"]], df_ds["class"], margins=True))

---
## Cell 5: Kiểm tra Kiến trúc `MedSAM-LoRA`, Tham số Khả huấn & Minh họa `Box Jittering ±10px`


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from data.unified_dataset import DEFAULT_ANNOTATION_FILES, UnifiedBrainMRIDataset
from models.medsam_lora import MedSamLoRA, jitter_boxes
from utils.losses import SegmentationLoss

device = "cuda" if torch.cuda.is_available() else "cpu"
model_check = MedSamLoRA.from_checkpoint(
    checkpoint_path("medsam"),
    lora_r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    device="cpu",
    verify=True,
)
counts = model_check.param_counts()
df_params = pd.DataFrame([
    {"Thành phần": "Image Encoder (đóng băng)", "Số tham số": counts["image_encoder_frozen_params"], "Trạng thái": "Frozen"},
    {"Thành phần": f"LoRA (r={LORA_R}, α={LORA_ALPHA}, 12 khối Q/V)", "Số tham số": counts["lora_params"], "Trạng thái": "Trainable"},
    {"Thành phần": "Mask Decoder (full)", "Số tham số": counts["mask_decoder_params"], "Trạng thái": "Trainable"},
    {"Thành phần": "TỔNG KHẢ HUẤN", "Số tham số": counts["trainable"], "Trạng thái": f"{counts['trainable_pct']:.2f}% / {counts['total']:,}"},
])
display(df_params)

# Kiểm tra nhanh hàm mất mát L_Seg = 0.5*Dice + 0.5*Focal/BCE
crit_check = SegmentationLoss(mode=LOSS_MODE, dice_weight=DICE_WEIGHT, pixel_weight=PIXEL_WEIGHT)
dummy_loss = crit_check(torch.randn(2, 1, 64, 64), torch.randint(0, 2, (2, 1, 64, 64)).float())
print(f"Kiểm tra SegmentationLoss ({LOSS_MODE.upper()}): total={float(dummy_loss['loss']):.4f} "
      f"(dice={float(dummy_loss['dice_loss']):.4f}, pixel={float(dummy_loss['pixel_loss']):.4f})")
del model_check

# Minh họa 4 mẫu kèm GT Mask & Box Jittering ±10px
vis_ds = UnifiedBrainMRIDataset(list(DEFAULT_ANNOTATION_FILES), train=False, datasets=TARGET_DATASETS, split="train")
vis_ds.records = [r for r in vis_ds.records if bool(r.get("has_mask")) and int(r.get("has_tumor", 0)) == 1]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
step = max(1, len(vis_ds) // 4)
for idx, ax in enumerate(axes):
    sample = vis_ds[min(idx * step, len(vis_ds) - 1)]
    img = sample["image_1024"].permute(1, 2, 0).numpy()
    msk = sample["gt_mask"].numpy()
    gt_b = sample["gt_box"].unsqueeze(0)
    jit_b = jitter_boxes(gt_b, max_px=BOX_JITTER, img_size=1024)[0].numpy()
    gt_b = gt_b[0].numpy()

    ax.imshow(np.clip(img, 0, 1))
    ax.imshow(np.ma.masked_where(msk < 0.5, msk), cmap="autumn", alpha=0.4)
    ax.add_patch(patches.Rectangle((gt_b[0], gt_b[1]), gt_b[2] - gt_b[0], gt_b[3] - gt_b[1],
                                   lw=2, ec="lime", fc="none", label="GT Box"))
    ax.add_patch(patches.Rectangle((jit_b[0], jit_b[1]), jit_b[2] - jit_b[0], jit_b[3] - jit_b[1],
                                   lw=1.8, ec="red", ls="--", fc="none", label=f"Jitter ±{int(BOX_JITTER)}px"))
    ax.set_title(f"{sample['dataset']} | class={sample['sub_class_label']}", fontsize=10)
    ax.axis("off")
    if idx == 0:
        ax.legend(loc="lower right", fontsize=8)
plt.suptitle(f"Minh họa Mask & Box Jittering ±{int(BOX_JITTER)}px (hệ 1024×1024)", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(RUN_DIR / "sample_box_jittering.png", dpi=150)
plt.show()

---
## Cell 6: Kiểm thử Nhanh (16 mẫu, 1 epoch — tùy chọn)


In [ ]:
def run_stream(cmd: list[str]):
    print("COMMAND:", " ".join(cmd), flush=True)
    env = dict(os.environ, PYTHONUNBUFFERED="1", PYTHONIOENCODING="utf-8")
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding="utf-8", errors="replace", bufsize=1, env=env) as proc:
        for line in proc.stdout:
            print(line, end="", flush=True)
        code = proc.wait()
    if code != 0:
        raise subprocess.CalledProcessError(code, cmd)

if RUN_SMOKE_TEST:
    smoke_dir = DRIVE_WORKDIR / "smoke_test"
    cmd_smoke = [
        sys.executable, "-m", "engine.train_medsam_lora",
        "--datasets", *TARGET_DATASETS,
        "--lora-r", str(LORA_R), "--lora-alpha", str(LORA_ALPHA), "--lora-dropout", str(LORA_DROPOUT),
        "--loss-mode", LOSS_MODE, "--dice-weight", str(DICE_WEIGHT), "--pixel-weight", str(PIXEL_WEIGHT),
        "--box-jitter", str(BOX_JITTER),
        "--limit", "16", "--epochs", "1",
        "--batch-size", str(BATCH_SIZE), "--grad-accum", "2",
        "--workers", str(NUM_WORKERS),
        "--out", str(smoke_dir),
    ]
    if "--resume" in (PROJECT_ROOT / "engine" / "train_medsam_lora.py").read_text(encoding="utf-8"):
        cmd_smoke.extend(["--resume", "never"])
    if torch.cuda.is_available():
        cmd_smoke.append("--amp")
    run_stream(cmd_smoke)
else:
    print("RUN_SMOKE_TEST = False (chuyển thẳng sang Cell 7).")

---
## Cell 7: Huấn luyện `MedSAM-LoRA` (`0.5 Dice + 0.5 Focal/BCE`, `Box Jittering ±10px`)


In [ ]:
def build_train_cmd(loss_mode: str, out_dir: Path) -> list[str]:
    cmd = [
        sys.executable, "-m", "engine.train_medsam_lora",
        "--datasets", *TARGET_DATASETS,
        "--lora-r", str(LORA_R),
        "--lora-alpha", str(LORA_ALPHA),
        "--lora-dropout", str(LORA_DROPOUT),
        "--loss-mode", loss_mode,
        "--dice-weight", str(DICE_WEIGHT),
        "--pixel-weight", str(PIXEL_WEIGHT),
        "--box-jitter", str(BOX_JITTER),
        "--epochs", str(EPOCHS),
        "--batch-size", str(BATCH_SIZE),
        "--grad-accum", str(GRAD_ACCUM),
        "--lr-lora", str(LR_LORA),
        "--lr-decoder", str(LR_DECODER),
        "--weight-decay", str(WEIGHT_DECAY),
        "--patience", str(PATIENCE),
        "--workers", str(NUM_WORKERS),
        "--seed", str(SEED),
        "--out", str(out_dir),
    ]
    if "--resume" in (PROJECT_ROOT / "engine" / "train_medsam_lora.py").read_text(encoding="utf-8"):
        cmd.extend(["--resume", "auto"])
    if torch.cuda.is_available():
        cmd.append("--amp")
    return cmd


print(f"=== Bắt đầu huấn luyện MedSAM-LoRA ({LOSS_MODE.upper()} + Dice, Box Jitter ±{BOX_JITTER}px) ===")
run_stream(build_train_cmd(LOSS_MODE, RUN_DIR))

if RUN_LOSS_COMPARISON:
    alt_mode = "bce" if LOSS_MODE == "focal" else "focal"
    alt_dir = DRIVE_WORKDIR / f"medsam_lora_{DS_TAG}_r{LORA_R}_{alt_mode}_j{int(BOX_JITTER)}"
    print(f"\n=== Chạy đối sánh hàm mất mát: {alt_mode.upper()} + Dice ===")
    run_stream(build_train_cmd(alt_mode, alt_dir))

---
## Cell 8: Đánh giá Đối sánh `Zero-shot MedSAM` vs `MedSAM-LoRA` trên Tập Test


In [ ]:
from collections import defaultdict
from torch.utils.data import DataLoader
from utils.metrics import SegmentationMeter

test_ds = UnifiedBrainMRIDataset(list(DEFAULT_ANNOTATION_FILES), train=False, datasets=TARGET_DATASETS, split="test")
test_ds.records = [
    r for r in test_ds.records
    if bool(r.get("has_mask")) and bool(r.get("mask_path")) and int(r.get("has_tumor", 0)) == 1
]
if EVAL_MAX_SAMPLES and len(test_ds.records) > EVAL_MAX_SAMPLES:
    rng = np.random.RandomState(SEED)
    keep = sorted(rng.choice(len(test_ds.records), EVAL_MAX_SAMPLES, replace=False))
    test_ds.records = [test_ds.records[i] for i in keep]

test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                         pin_memory=torch.cuda.is_available())
use_amp = torch.cuda.is_available()


@torch.no_grad()
def evaluate_model(model: MedSamLoRA, loader: DataLoader, desc: str) -> dict:
    model.eval()
    overall = SegmentationMeter()
    per_class = defaultdict(SegmentationMeter)
    sample_preds = []
    for batch in loader:
        images = batch["image_1024"].to(device, non_blocking=True)
        boxes = batch["gt_box"].to(device, non_blocking=True)
        masks = batch["gt_mask"].numpy() > 0.5
        labels = batch["sub_class_label"].tolist()
        with torch.autocast(device_type=device.split(":")[0], dtype=torch.float16, enabled=use_amp):
            logits = model(images, boxes, upsample=True)
        preds = (torch.sigmoid(logits.float()) > 0.5).squeeze(1).cpu().numpy()
        for idx, (pm, gm, lbl) in enumerate(zip(preds, masks, labels)):
            row = overall.update(pm, gm)
            cname = CLASS_NAMES.get(int(lbl), f"class_{lbl}")
            per_class[cname].update(pm, gm)
            if len(sample_preds) < 6:
                sample_preds.append({
                    "image": batch["image_1024"][idx].permute(1, 2, 0).numpy(),
                    "box": batch["gt_box"][idx].numpy(),
                    "gt": gm,
                    "pred": pm,
                    "dice": row["dice"],
                    "class": cname,
                })
    print(f"[{desc}] n={len(overall.rows)} | Dice={overall.summary()['dice']['mean']:.4f} | "
          f"IoU={overall.summary()['iou']['mean']:.4f} | HD95={overall.summary()['hd95']['mean']:.2f}px")
    return {
        "overall": overall.summary(),
        "per_class": {k: v.summary() for k, v in sorted(per_class.items())},
        "samples": sample_preds,
    }


# 1. Zero-shot MedSAM gốc (LoRA B=0 => đẳng thức chính xác với MedSAM pretrained gốc)
zeroshot_model = MedSamLoRA.from_checkpoint(
    checkpoint_path("medsam"), lora_r=LORA_R, lora_alpha=LORA_ALPHA, device=device, verify=True
)
res_zs = evaluate_model(zeroshot_model, test_loader, "Zero-shot MedSAM")

# 2. MedSAM-LoRA đã huấn luyện (nạp best_medsam_lora.pt)
best_ckpt_path = RUN_DIR / "best_medsam_lora.pt"
assert best_ckpt_path.is_file(), f"Không tìm thấy {best_ckpt_path}. Hãy chạy Bước 7 trước."
zeroshot_model.load_checkpoint(best_ckpt_path, map_location=device)
res_lora = evaluate_model(zeroshot_model, test_loader, f"MedSAM-LoRA (r={LORA_R})")

# Tổng hợp bảng đối sánh tổng thể & theo lớp
comp_rows = []
for name, res in [("MedSAM Zero-shot (Tuần 2)", res_zs["overall"]), (f"MedSAM-LoRA r={LORA_R} (Tuần 4)", res_lora["overall"])]:
    comp_rows.append({
        "Mô hình": name,
        "Số mẫu": res["n"],
        "Dice Mean (%)": round(res["dice"]["mean"] * 100, 2),
        "Dice Median (%)": round(res["dice"]["median"] * 100, 2),
        "IoU Mean (%)": round(res["iou"]["mean"] * 100, 2),
        "HD95 Mean (px)": round(res["hd95"]["mean"], 2),
    })
df_comp = pd.DataFrame(comp_rows)
delta_dice = comp_rows[1]["Dice Mean (%)"] - comp_rows[0]["Dice Mean (%)"]
delta_hd95 = comp_rows[1]["HD95 Mean (px)"] - comp_rows[0]["HD95 Mean (px)"]
print(f"\nCải thiện MedSAM-LoRA vs Zero-shot: ΔDice = {delta_dice:+.2f}% | ΔHD95 = {delta_hd95:+.2f} px")
display(df_comp)

cls_rows = []
for cname in sorted(res_lora["per_class"]):
    zs_c = res_zs["per_class"].get(cname, {})
    lr_c = res_lora["per_class"][cname]
    cls_rows.append({
        "Nhóm bệnh": cname,
        "Số mẫu": lr_c["n"],
        "Zero-shot Dice (%)": round(zs_c.get("dice", {}).get("mean", 0.0) * 100, 2),
        "MedSAM-LoRA Dice (%)": round(lr_c["dice"]["mean"] * 100, 2),
        "ΔDice (%)": round((lr_c["dice"]["mean"] - zs_c.get("dice", {}).get("mean", 0.0)) * 100, 2),
        "MedSAM-LoRA IoU (%)": round(lr_c["iou"]["mean"] * 100, 2),
        "MedSAM-LoRA HD95 (px)": round(lr_c["hd95"]["mean"], 2),
    })
df_cls = pd.DataFrame(cls_rows)
display(df_cls)

df_comp.to_csv(RUN_DIR / "zeroshot_vs_lora_overall.csv", index=False)
df_cls.to_csv(RUN_DIR / "zeroshot_vs_lora_per_class.csv", index=False)
(RUN_DIR / "zeroshot_vs_lora_summary.json").write_text(
    json.dumps({"overall": comp_rows, "per_class": cls_rows}, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
print(f"Đã lưu bảng đối sánh tại: {RUN_DIR}")

---
## Cell 9: Đường cong Hội tụ & Minh họa Định tính (`MRI | Ground Truth | Zero-shot | MedSAM-LoRA`)


In [ ]:
# 1. Đường cong hội tụ từ history.csv
hist_path = RUN_DIR / "history.csv"
if hist_path.is_file():
    hist = pd.read_csv(hist_path)
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

    axes[0].plot(hist["epoch"], hist["train_loss"], marker="o", label="Train Total Loss")
    axes[0].plot(hist["epoch"], hist["val_loss"], marker="s", label="Val Total Loss")
    axes[0].set_title(f"Hội tụ Loss (0.5 Dice + 0.5 {LOSS_MODE.upper()})")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].grid(alpha=0.3)
    axes[0].legend()

    axes[1].plot(hist["epoch"], hist["train_dice"] * 100, marker="o", label="Train Dice (%)")
    axes[1].plot(hist["epoch"], hist["val_dice"] * 100, marker="s", label="Val Dice (%)")
    axes[1].plot(hist["epoch"], hist["val_iou"] * 100, marker="^", label="Val IoU (%)")
    axes[1].set_title("Dice & IoU theo Epoch")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Score (%)")
    axes[1].grid(alpha=0.3)
    axes[1].legend()

    axes[2].plot(hist["epoch"], hist["val_hd95"], marker="d", color="crimson", label="Val HD95 (px)")
    axes[2].set_title("Sai số biên HD95 trên Validation")
    axes[2].set_xlabel("Epoch")
    axes[2].set_ylabel("HD95 (px)")
    axes[2].grid(alpha=0.3)
    axes[2].legend()

    plt.tight_layout()
    plt.savefig(RUN_DIR / "training_curves.png", dpi=180)
    plt.show()

# 2. Lưới ảnh so sánh định tính: MRI + GT Box | Ground Truth | Zero-shot MedSAM | MedSAM-LoRA
n_show = min(len(res_zs["samples"]), len(res_lora["samples"]), 6)
if n_show > 0:
    fig, axes = plt.subplots(n_show, 4, figsize=(14, 3.3 * n_show))
    if n_show == 1:
        axes = np.expand_dims(axes, axis=0)
    for i in range(n_show):
        s_zs = res_zs["samples"][i]
        s_lr = res_lora["samples"][i]
        img = np.clip(s_lr["image"], 0, 1)
        b = s_lr["box"]

        axes[i, 0].imshow(img)
        axes[i, 0].add_patch(patches.Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1], lw=2, ec="lime", fc="none"))
        axes[i, 0].set_title(f"MRI + Box ({s_lr['class']})", fontsize=9)

        axes[i, 1].imshow(img)
        axes[i, 1].imshow(np.ma.masked_where(~s_lr["gt"], s_lr["gt"]), cmap="Greens", alpha=0.55)
        axes[i, 1].set_title("Ground Truth Mask", fontsize=9)

        axes[i, 2].imshow(img)
        axes[i, 2].imshow(np.ma.masked_where(~s_zs["pred"], s_zs["pred"]), cmap="Oranges", alpha=0.55)
        axes[i, 2].set_title(f"Zero-shot MedSAM (Dice={s_zs['dice']*100:.1f}%)", fontsize=9)

        axes[i, 3].imshow(img)
        axes[i, 3].imshow(np.ma.masked_where(~s_lr["pred"], s_lr["pred"]), cmap="cool", alpha=0.55)
        axes[i, 3].set_title(f"MedSAM-LoRA r={LORA_R} (Dice={s_lr['dice']*100:.1f}%)", fontsize=9)

        for j in range(4):
            axes[i, j].axis("off")

    plt.suptitle("So sánh định tính: Zero-shot MedSAM vs MedSAM-LoRA", fontsize=13, fontweight="bold", y=1.005)
    plt.tight_layout()
    plt.savefig(RUN_DIR / "qualitative_zeroshot_vs_lora.png", dpi=160, bbox_inches="tight")
    plt.show()
    print(f"Hoàn tất! Toàn bộ checkpoint, bảng số liệu và hình ảnh đã lưu tại: {RUN_DIR}")